# Lezione 09 - Pattern di progettazione metacognitiva


## Configurazione

Questo notebook dimostra il pattern di progettazione Metacognition utilizzando il Microsoft Agent Framework.

**Prerequisiti:**
- Distribuzione di Azure OpenAI configurata tramite variabili d'ambiente
- Azure CLI autenticata (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Che cos'è la metacognizione?

La metacognizione è **riflettere sul pensare**. Nel contesto degli agenti di intelligenza artificiale, significa costruire agenti che possono:

- **Autoriflettere** sui propri risultati e sul processo di ragionamento
- **Rilevare errori** e recuperare elegantemente invece di fallire silenziosamente
- **Valutare** se le loro risposte sono complete e utili
- **Adattare** la propria strategia quando un approccio iniziale non funziona (e.g., ricorrendo a un sistema di backup)

Un agente metacognitivo non si limita a rispondere alle domande — monitora le proprie prestazioni e si adatta al volo.


## Strumenti primari e di riserva

Un modello comune di metacognizione è la **strategia di fallback**. L'agente prova prima uno strumento primario; se fallisce (ad es., un errore 404), l'agente riconosce il fallimento e passa in modo trasparente a uno strumento di riserva.

Questo rispecchia i sistemi del mondo reale in cui i servizi primari possono essere non disponibili e gli agenti devono autodiagnosticare il problema prima di scegliere un percorso alternativo.

Di seguito definiamo due strumenti per la ricerca di voli:
- **Primario** — copre Parigi, Tokyo e Barcellona
- **Di riserva** — copre Berlino, Sydney e New York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agente autoriflessivo con recupero dagli errori

L'agente qui sotto è istruito a provare prima il sistema di volo primario, riconoscere i guasti e ricorrere in modo trasparente al sistema di backup. Dopo ogni risposta si autovaluta brevemente per verificare se ha risposto completamente alla domanda dell'utente.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Modello di autovalutazione

Un altro aspetto della metacognizione è **l'autovalutazione**: un agente separato (o lo stesso agente in un secondo passaggio) esamina una risposta per completezza, accuratezza e utilità.

Di seguito creiamo un agente `ResponseEvaluator` che assegna un punteggio alle risposte del `travel-agent` su tre dimensioni.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Summary

In questa lezione hai imparato come costruire **agenti metacognitivi** usando il Microsoft Agent Framework:

- **Autoriflessione**: Agenti che monitorano il proprio ragionamento e comunicano in modo trasparente ciò che è successo.
- **Recupero dagli errori con fallback**: Un pattern strumento primario + di backup in cui l'agente rileva i fallimenti (ad es., errori 404) e prova automaticamente una fonte alternativa.
- **Autovalutazione**: Un agente valutatore separato che assegna un punteggio alle risposte per completezza, accuratezza e utilità.

Questi schemi rendono gli agenti più robusti, trasparenti e affidabili — qualità critiche per le distribuzioni in produzione.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Dichiarazione di non responsabilità:
Questo documento è stato tradotto mediante il servizio di traduzione basato su IA Co-op Translator (https://github.com/Azure/co-op-translator). Sebbene ci impegniamo a garantire la massima accuratezza, si prega di notare che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella sua lingua originale deve essere considerato la fonte autorevole. Per informazioni critiche è raccomandata una traduzione professionale effettuata da un traduttore umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall'uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
